# Phase 4 - Notebook 05: Feed-forward Training & Loss Design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/05_training_loss_design.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the training data pipeline for feed-forward 3DGS (RE10K, ACID)
2. Implement the photometric loss functions (L1, SSIM, LPIPS)
3. Understand view sampling strategies for training
4. Build a complete training loop for feed-forward Gaussian prediction
5. Implement evaluation metrics (PSNR, SSIM, LPIPS)

**Estimated Time**: 75 minutes

**Prerequisites**: Notebooks 03-04 (MVSplat & pixelSplat architectures)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")

## 1. Training Paradigm: Feed-forward vs Per-scene

### Phase 1 (Per-scene optimization)
```
One scene → Initialize Gaussians → Optimize 30K iterations → Done
  - Training = inference
  - No generalization
  - Loss: render same scene views
```

### Phase 4 (Feed-forward training)
```
1000s of scenes → Train network → Inference on NEW scenes
  - Training on a dataset of scenes
  - Generalization is the goal
  - Loss: render NOVEL views (not the input views!)
```

### Training Data Format

Each training sample consists of:

| Component | Shape | Description |
|-----------|-------|-------------|
| Input views | 2 x [3, H, W] | Context images (fed to model) |
| Target views | 1-3 x [3, H, W] | Novel views for supervision |
| Camera intrinsics | [3, 3] | Focal length, principal point |
| Camera poses | N x [4, 4] | Camera-to-world for each view |
| (optional) Depth | [1, H, W] | Ground truth depth maps |

In [ ]:
# Visualize the training paradigm

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: Per-scene optimization (Phase 1)
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('Phase 1: Per-scene Optimization', fontsize=13, fontweight='bold')

# Draw scene images
for i in range(5):
    rect = plt.Rectangle((0.5 + i * 1.8, 5), 1.2, 1.5, 
                          facecolor='#E3F2FD', edgecolor='#1565C0', lw=1.5)
    ax.add_patch(rect)
    ax.text(1.1 + i * 1.8, 5.75, f'View {i+1}', ha='center', fontsize=7)

ax.annotate('', xy=(5, 4.5), xytext=(5, 5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#1565C0'))
rect = plt.Rectangle((2.5, 2.5), 5, 1.5, facecolor='#BBDEFB', edgecolor='#1565C0', lw=2)
ax.add_patch(rect)
ax.text(5, 3.5, 'Optimize Gaussians', ha='center', fontsize=11, fontweight='bold')
ax.text(5, 2.9, '30K iters on THIS scene only', ha='center', fontsize=8, style='italic')

ax.annotate('', xy=(5, 2), xytext=(5, 2.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#1565C0'))
ax.text(5, 1.5, 'Render only THIS scene', ha='center', fontsize=10, color='#D32F2F')
ax.text(5, 0.8, 'No generalization to new scenes', ha='center', fontsize=8,
        style='italic', color='#666')

# Right: Feed-forward training (Phase 4)
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('Phase 4: Feed-forward Training', fontsize=13, fontweight='bold')

# Multiple scenes
scene_colors = ['#E8F5E9', '#FFF3E0', '#F3E5F5']
for j, (color, label) in enumerate(zip(scene_colors, ['Scene A', 'Scene B', 'Scene C'])):
    for i in range(2):
        rect = plt.Rectangle((0.3 + i * 1.2 + j * 3.2, 5.8), 0.9, 1.1,
                              facecolor=color, edgecolor='#333', lw=1)
        ax.add_patch(rect)
    ax.text(1.2 + j * 3.2, 5.3, label, ha='center', fontsize=7, style='italic')

ax.text(5, 7.5, '1000s of scenes from RE10K/ACID', ha='center', fontsize=9)

ax.annotate('', xy=(5, 4.7), xytext=(5, 5.3),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2E7D32'))
rect = plt.Rectangle((2, 3.2), 6, 1.5, facecolor='#C8E6C9', edgecolor='#2E7D32', lw=2)
ax.add_patch(rect)
ax.text(5, 4.2, 'Train Neural Network', ha='center', fontsize=11, fontweight='bold')
ax.text(5, 3.6, 'Across ALL scenes, 300K iters', ha='center', fontsize=8, style='italic')

ax.annotate('', xy=(5, 2.5), xytext=(5, 3.2),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2E7D32'))
ax.text(5, 1.8, 'Render ANY new scene', ha='center', fontsize=10, color='#2E7D32')
ax.text(5, 1.1, 'Generalization to unseen scenes!', ha='center', fontsize=8,
        style='italic', color='#666')

plt.tight_layout()
plt.show()

## 2. Datasets for Feed-forward 3DGS

### 2.1 Primary Datasets

| Dataset | Type | Scenes | Source | Used by |
|---------|------|--------|--------|--------|
| **RE10K** | Indoor/outdoor | ~10K videos | YouTube | MVSplat, pixelSplat |
| **ACID** | Outdoor aerial | ~11K sequences | Aerial video | MVSplat |
| **DTU** | Objects | 124 scans | Lab captures | Evaluation |
| **ScanNet** | Indoor rooms | 1513 scans | RGB-D sensor | With depth |

### 2.2 View Sampling Strategy

For each training step, we sample:
1. **Context views** (input to model): 2 images with moderate baseline
2. **Target views** (for supervision): 1-3 novel viewpoints

The baseline between context views is important:
- Too small: insufficient parallax for depth estimation
- Too large: difficult matching, many occlusions

In [ ]:
class SyntheticSceneDataset(torch.utils.data.Dataset):
    """
    Synthetic dataset that simulates RE10K-style training data.
    
    Generates random scenes with:
    - Random textured planes at different depths
    - Random camera poses with controlled baseline
    - Rendered images from multiple viewpoints
    """

    def __init__(self, num_scenes=100, image_size=64, num_context=2, num_target=1):
        self.num_scenes = num_scenes
        self.image_size = image_size
        self.num_context = num_context
        self.num_target = num_target

        # Pre-generate random scenes
        self.fx = self.fy = 50.0
        self.cx = self.cy = image_size / 2.0

    def __len__(self):
        return self.num_scenes

    def _generate_camera_pose(self, idx, baseline_scale=0.3):
        """Generate camera pose with controlled baseline."""
        angle = idx * 0.15 * baseline_scale  # Small rotation
        tx = idx * baseline_scale
        R = torch.tensor([
            [np.cos(angle), 0, np.sin(angle)],
            [0, 1, 0],
            [-np.sin(angle), 0, np.cos(angle)],
        ], dtype=torch.float32)
        pose = torch.eye(4)
        pose[:3, :3] = R
        pose[:3, 3] = torch.tensor([tx, 0.0, 0.0])
        return pose

    def _render_synthetic(self, pose, scene_seed):
        """Render a synthetic image from the given pose."""
        H = W = self.image_size
        torch.manual_seed(scene_seed)

        # Create a simple textured depth scene
        u = torch.linspace(-1, 1, W)
        v = torch.linspace(-1, 1, H)
        VV, UU = torch.meshgrid(v, u, indexing='ij')

        # Depth: random plane with bumps
        freq = torch.rand(1) * 3 + 1
        depth = 5.0 + torch.sin(UU * freq * np.pi) * torch.cos(VV * freq * np.pi)

        # Simple texture: depends on pose (different per view)
        tx = pose[0, 3].item()
        image = torch.stack([
            0.5 + 0.3 * torch.sin(UU * 5 + tx * 2),
            0.5 + 0.3 * torch.cos(VV * 5 + tx),
            0.5 + 0.2 * torch.sin((UU + VV) * 3 + tx * 3),
        ], dim=0)  # [3, H, W]
        image = image.clamp(0, 1)

        return image, depth

    def __getitem__(self, idx):
        scene_seed = idx * 1000
        total_views = self.num_context + self.num_target

        images = []
        poses = []
        depths = []

        for v in range(total_views):
            pose = self._generate_camera_pose(v)
            image, depth = self._render_synthetic(pose, scene_seed)
            images.append(image)
            poses.append(pose)
            depths.append(depth)

        K = torch.tensor([
            [self.fx, 0, self.cx],
            [0, self.fy, self.cy],
            [0, 0, 1],
        ], dtype=torch.float32)

        return {
            'context_images': torch.stack(images[:self.num_context]),  # [Nc, 3, H, W]
            'target_images': torch.stack(images[self.num_context:]),   # [Nt, 3, H, W]
            'context_poses': torch.stack(poses[:self.num_context]),    # [Nc, 4, 4]
            'target_poses': torch.stack(poses[self.num_context:]),     # [Nt, 4, 4]
            'K': K,                                                    # [3, 3]
            'context_depths': torch.stack(depths[:self.num_context]),  # [Nc, H, W]
        }


# Create dataset
dataset = SyntheticSceneDataset(num_scenes=50, image_size=64)
sample = dataset[0]

print("Training sample contents:")
for key, val in sample.items():
    print(f"  {key:20s}: shape={list(val.shape)}, dtype={val.dtype}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ['Context View 0', 'Context View 1', 'Target View']
all_imgs = [sample['context_images'][0], sample['context_images'][1],
            sample['target_images'][0]]

for ax, img, title in zip(axes, all_imgs, titles):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Training Sample: 2 context views (input) + 1 target view (supervision)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Loss Functions

### 3.1 Overview

Feed-forward 3DGS methods are trained with **photometric loss** on novel views:

$$\mathcal{L}_{total} = \lambda_1 \mathcal{L}_{1} + \lambda_{ssim} \mathcal{L}_{SSIM} + \lambda_{lpips} \mathcal{L}_{LPIPS}$$

The training process:
1. Feed context views through the model to predict Gaussians
2. **Render** the Gaussians from the **target viewpoint**
3. Compare the rendered image to the target ground truth
4. Backpropagate through rendering, prediction, and encoder

### 3.2 L1 Loss (Pixel-level)

In [ ]:
def l1_loss(predicted, target):
    """
    L1 loss: Mean absolute error per pixel.
    
    Simple and effective for pixel-level color matching.
    Less sensitive to outliers than L2.
    """
    return (predicted - target).abs().mean()


def l2_loss(predicted, target):
    """L2 (MSE) loss for comparison."""
    return (predicted - target).pow(2).mean()


# Demonstrate L1 vs L2 sensitivity to outliers
torch.manual_seed(42)
H, W = 64, 64
target = torch.rand(1, 3, H, W)

# Create predictions with different noise levels
noise_levels = [0.01, 0.05, 0.1, 0.2, 0.5]
l1_losses = []
l2_losses = []

for noise in noise_levels:
    pred = target + torch.randn_like(target) * noise
    pred = pred.clamp(0, 1)
    l1_losses.append(l1_loss(pred, target).item())
    l2_losses.append(l2_loss(pred, target).item())

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(noise_levels, l1_losses, 'o-', label='L1 loss', lw=2, markersize=8)
ax.plot(noise_levels, l2_losses, 's-', label='L2 loss', lw=2, markersize=8)
ax.set_xlabel('Noise Level (std)', fontsize=11)
ax.set_ylabel('Loss Value', fontsize=11)
ax.set_title('L1 vs L2 Loss: Response to Noise', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("L1 loss is preferred in practice because:")
print("  - Less sensitive to outliers (more robust)")
print("  - Produces sharper images (L2 tends to blur)")
print("  - Linear gradient (stable training)")

### 3.3 SSIM Loss (Structural Similarity)

SSIM measures perceptual similarity by comparing **luminance**, **contrast**, and **structure** between image patches:

$$SSIM(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

The loss is: $\mathcal{L}_{SSIM} = 1 - SSIM(x, y)$

In [ ]:
def compute_ssim(img1, img2, window_size=11):
    """
    Compute SSIM between two images.
    
    Args:
        img1, img2: [B, C, H, W] in range [0, 1]
        window_size: size of Gaussian window
    
    Returns:
        ssim_map: [B, 1, H, W] per-pixel SSIM
    """
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    C = img1.shape[1]

    # Create Gaussian window
    sigma = 1.5
    coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window = g.unsqueeze(1) * g.unsqueeze(0)  # 2D Gaussian
    window = window.unsqueeze(0).unsqueeze(0)  # [1, 1, ws, ws]
    window = window.expand(C, -1, -1, -1).to(img1.device)

    pad = window_size // 2

    mu1 = F.conv2d(img1, window, padding=pad, groups=C)
    mu2 = F.conv2d(img2, window, padding=pad, groups=C)

    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu12 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=pad, groups=C) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=pad, groups=C) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=pad, groups=C) - mu12

    ssim_map = ((2 * mu12 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    return ssim_map.mean(dim=1, keepdim=True)  # Average over channels


def ssim_loss(predicted, target):
    """SSIM loss = 1 - SSIM (lower is better)."""
    return 1.0 - compute_ssim(predicted, target).mean()


# Compare L1 vs SSIM for different types of distortions
torch.manual_seed(42)
H, W = 64, 64
target = torch.rand(1, 3, H, W)

# Create different distortion types
distortions = {
    'Noise': target + torch.randn_like(target) * 0.1,
    'Blur': F.avg_pool2d(F.pad(target, [2]*4, mode='reflect'), 5, stride=1),
    'Shift': torch.roll(target, shifts=3, dims=3),
    'Contrast': target * 0.5 + 0.25,
}

fig, axes = plt.subplots(2, 5, figsize=(20, 7))

# Show images
axes[0, 0].imshow(target[0].permute(1, 2, 0).numpy())
axes[0, 0].set_title('Target', fontsize=10, fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for i, (name, distorted) in enumerate(distortions.items()):
    distorted = distorted.clamp(0, 1)
    l1_val = l1_loss(distorted, target).item()
    ssim_val = compute_ssim(distorted, target).mean().item()

    # Show distorted image
    ax = axes[0, i + 1]
    ax.imshow(distorted[0].permute(1, 2, 0).numpy())
    ax.set_title(f'{name}', fontsize=10, fontweight='bold')
    ax.axis('off')

    # Show SSIM map
    ax = axes[1, i + 1]
    ssim_map = compute_ssim(distorted, target)[0, 0].detach().numpy()
    im = ax.imshow(ssim_map, cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_title(f'L1={l1_val:.3f}, SSIM={ssim_val:.3f}', fontsize=9)
    ax.axis('off')

plt.suptitle('L1 vs SSIM: Different distortion types',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("SSIM captures structural similarity that L1 misses.")
print("Using both gives complementary supervision signals.")

### 3.4 LPIPS Loss (Perceptual)

LPIPS (Learned Perceptual Image Patch Similarity) compares images in deep feature space:

$$\mathcal{L}_{LPIPS} = \sum_l w_l \cdot \| \phi_l(\hat{I}) - \phi_l(I) \|_2^2$$

where $\phi_l$ are features from a pretrained VGG/AlexNet at layer $l$.

Since LPIPS requires a pretrained VGG network, we'll implement a simplified version here.

In [ ]:
class SimplifiedPerceptualLoss(nn.Module):
    """
    Simplified perceptual loss using random CNN features.
    
    In practice, this uses pretrained VGG features.
    This educational version shows the concept.
    """

    def __init__(self):
        super().__init__()
        # Multi-scale feature extractor (simulates VGG layers)
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(3, 32, 3, padding=1),
                nn.ReLU(inplace=True),
            ),
            nn.Sequential(
                nn.Conv2d(32, 64, 3, stride=2, padding=1),
                nn.ReLU(inplace=True),
            ),
            nn.Sequential(
                nn.Conv2d(64, 128, 3, stride=2, padding=1),
                nn.ReLU(inplace=True),
            ),
        ])

        # Freeze weights (simulates pretrained network)
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, predicted, target):
        """Compute perceptual loss."""
        loss = 0.0
        x_pred = predicted
        x_target = target

        for layer in self.layers:
            x_pred = layer(x_pred)
            x_target = layer(x_target)

            # Normalize features
            x_pred_norm = F.normalize(x_pred, dim=1)
            x_target_norm = F.normalize(x_target, dim=1)

            # Feature distance
            loss += (x_pred_norm - x_target_norm).pow(2).mean()

        return loss / len(self.layers)


perceptual_loss = SimplifiedPerceptualLoss()

# Compare different loss components
torch.manual_seed(42)
pred = target + torch.randn_like(target) * 0.1
pred = pred.clamp(0, 1)

print("Loss components for noisy prediction:")
print(f"  L1 loss:         {l1_loss(pred, target):.4f}")
print(f"  SSIM loss:       {ssim_loss(pred, target):.4f}")
print(f"  Perceptual loss: {perceptual_loss(pred, target):.4f}")

### 3.5 Combined Loss

In practice, all three losses are combined:

In [ ]:
class FeedForwardLoss(nn.Module):
    """
    Combined loss for feed-forward 3DGS training.
    
    L = lambda_l1 * L1 + lambda_ssim * (1 - SSIM) + lambda_lpips * LPIPS
    
    Typical weights:
    - MVSplat:   lambda_l1=1.0, lambda_ssim=0.2, lambda_lpips=0.05
    - pixelSplat: lambda_l1=1.0, lambda_ssim=0.2, lambda_lpips=0.05
    """

    def __init__(self, lambda_l1=1.0, lambda_ssim=0.2, lambda_lpips=0.05):
        super().__init__()
        self.lambda_l1 = lambda_l1
        self.lambda_ssim = lambda_ssim
        self.lambda_lpips = lambda_lpips
        self.perceptual = SimplifiedPerceptualLoss()

    def forward(self, rendered, target):
        """
        Compute combined loss.
        
        Args:
            rendered: [B, 3, H, W] rendered novel view
            target: [B, 3, H, W] ground truth novel view
        
        Returns:
            total_loss, loss_dict
        """
        loss_l1 = l1_loss(rendered, target)
        loss_ssim = ssim_loss(rendered, target)
        loss_lpips = self.perceptual(rendered, target)

        total = (self.lambda_l1 * loss_l1 +
                 self.lambda_ssim * loss_ssim +
                 self.lambda_lpips * loss_lpips)

        return total, {
            'l1': loss_l1.item(),
            'ssim': loss_ssim.item(),
            'lpips': loss_lpips.item(),
            'total': total.item(),
        }


loss_fn = FeedForwardLoss(lambda_l1=1.0, lambda_ssim=0.2, lambda_lpips=0.05)

# Test
rendered = target + torch.randn_like(target) * 0.15
rendered = rendered.clamp(0, 1)

total, losses = loss_fn(rendered, target)
print("Combined loss breakdown:")
for name, val in losses.items():
    print(f"  {name:8s}: {val:.4f}")

## 4. Training Loop

### 4.1 Complete Training Step

The training loop for feed-forward 3DGS:

```
for each batch:
    1. Extract context views and target views
    2. Forward pass: context images → predict Gaussians
    3. Render: Gaussians → image at target viewpoint
    4. Loss: compare rendered vs ground truth target
    5. Backpropagate through entire pipeline
    6. Update weights
```

The gradient flows all the way from the rendered image back through:
- Differentiable rendering (Phase 1)
- Gaussian parameters
- Prediction heads
- Cost Volume / Attention
- Feature encoder

In [ ]:
class SimpleRenderer(nn.Module):
    """
    Very simplified renderer for training demonstration.
    
    In practice, this would use gsplat or diff-gaussian-rasterization
    to render Gaussians from a novel viewpoint. Here we simulate
    the rendering with a simple projection.
    """

    def __init__(self, H, W):
        super().__init__()
        self.H = H
        self.W = W

    def forward(self, gaussians_dict, K, target_pose):
        """
        Simplified rendering: splat Gaussian colors to image.
        
        Args:
            gaussians_dict: dict with positions [B,N,3], colors [B,N,3],
                           opacities [B,N,1]
            K: [B, 3, 3] intrinsics
            target_pose: [B, 4, 4] target camera pose
        
        Returns:
            rendered: [B, 3, H, W]
        """
        B = K.shape[0]
        positions = gaussians_dict['positions']  # [B, N, 3]
        colors = gaussians_dict['colors']        # [B, N, 3]
        opacities = gaussians_dict['opacities']  # [B, N, 1]

        # Transform to target camera frame
        R = target_pose[:, :3, :3]  # [B, 3, 3]
        t = target_pose[:, :3, 3:]  # [B, 3, 1]
        R_inv = R.transpose(1, 2)
        t_inv = -torch.bmm(R_inv, t)

        # Transform positions
        pos_cam = torch.bmm(positions, R_inv.transpose(1, 2)) + t_inv.transpose(1, 2)

        # Project to image
        proj = torch.bmm(pos_cam, K.transpose(1, 2))  # [B, N, 3]
        u = proj[:, :, 0] / (proj[:, :, 2] + 1e-8)  # [B, N]
        v = proj[:, :, 1] / (proj[:, :, 2] + 1e-8)
        z = proj[:, :, 2]  # depth

        # Simple splatting: accumulate weighted colors
        rendered = torch.zeros(B, 3, self.H, self.W, device=K.device)
        weight_sum = torch.zeros(B, 1, self.H, self.W, device=K.device)

        # Quantize pixel locations
        u_int = u.long().clamp(0, self.W - 1)
        v_int = v.long().clamp(0, self.H - 1)

        valid = (z > 0.1) & (u >= 0) & (u < self.W) & (v >= 0) & (v < self.H)

        for b in range(B):
            mask = valid[b]
            if mask.sum() == 0:
                continue
            ui = u_int[b, mask]
            vi = v_int[b, mask]
            c = colors[b, mask]      # [M, 3]
            a = opacities[b, mask]   # [M, 1]

            # Weighted accumulation
            for ch in range(3):
                rendered[b, ch].index_put_(
                    (vi, ui), c[:, ch] * a.squeeze(-1), accumulate=True
                )
            weight_sum[b, 0].index_put_(
                (vi, ui), a.squeeze(-1), accumulate=True
            )

        # Normalize
        rendered = rendered / (weight_sum + 1e-8)
        rendered = rendered.clamp(0, 1)

        return rendered


renderer = SimpleRenderer(H=64, W=64)
print("Simple renderer created for training demonstration.")
print("In practice, use gsplat or diff-gaussian-rasterization.")

In [ ]:
from src.feedforward.gaussian_predictor import GaussianPredictionHeads
from src.feedforward.pixel_aligned import unproject_depth_to_3d


class TinyFeedForwardModel(nn.Module):
    """
    Tiny feed-forward model for training demonstration.
    Minimal version to show the training loop concept.
    """

    def __init__(self, feature_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, feature_dim, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim, feature_dim, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.heads = GaussianPredictionHeads(
            in_channels=feature_dim,
            hidden_channels=16,
            depth_mode='regression',
            covariance_mode='3d',
        )

    def forward(self, context_images, K):
        """Predict Gaussians from context images."""
        B = context_images.shape[0]
        img0 = context_images[:, 0]  # First context view [B, 3, H, W]

        features = self.encoder(img0)
        predictions = self.heads(features)

        # Create Gaussians
        depth = predictions['depth']  # [B, 1, H, W]
        positions = unproject_depth_to_3d(depth, K)  # [B, H, W, 3]
        _, _, H, W = depth.shape
        N = H * W

        gaussians = {
            'positions': positions.reshape(B, N, 3),
            'colors': img0.flatten(2).permute(0, 2, 1),  # [B, N, 3]
            'opacities': predictions['opacities'].flatten(2).permute(0, 2, 1),
            'scales': predictions['scales'].flatten(2).permute(0, 2, 1),
            'rotations': predictions['rotations'].flatten(2).permute(0, 2, 1),
        }

        return gaussians, predictions


# Training loop
model = TinyFeedForwardModel(feature_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = FeedForwardLoss(lambda_l1=1.0, lambda_ssim=0.2, lambda_lpips=0.0)
renderer = SimpleRenderer(H=64, W=64)

# Training
dataset = SyntheticSceneDataset(num_scenes=50, image_size=64)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

num_epochs = 5
train_losses = []

print("Training feed-forward model...")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print()

for epoch in range(num_epochs):
    epoch_losses = []
    for batch_idx, batch in enumerate(dataloader):
        optimizer.zero_grad()

        # 1. Extract data
        context = batch['context_images']  # [B, Nc, 3, H, W]
        target = batch['target_images'][:, 0]  # [B, 3, H, W]
        K = batch['K']  # [B, 3, 3]
        target_pose = batch['target_poses'][:, 0]  # [B, 4, 4]

        # 2. Forward: predict Gaussians from context views
        gaussians, preds = model(context, K)

        # 3. Render from target viewpoint
        rendered = renderer(gaussians, K, target_pose)

        # 4. Compute loss
        total_loss, loss_dict = loss_fn(rendered, target)

        # 5. Backpropagate
        total_loss.backward()
        optimizer.step()

        epoch_losses.append(loss_dict['total'])

    avg_loss = np.mean(epoch_losses)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs}: loss = {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Plot training curve and visualize results

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training curve
ax = axes[0]
ax.plot(range(1, len(train_losses) + 1), train_losses, 'o-', lw=2, color='#1565C0')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Training Loss', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Show a rendered example
model.eval()
with torch.no_grad():
    sample = dataset[0]
    context = sample['context_images'].unsqueeze(0)
    target_img = sample['target_images'][0]
    K_test = sample['K'].unsqueeze(0)
    target_p = sample['target_poses'][0].unsqueeze(0)

    gaussians, _ = model(context, K_test)
    rendered = renderer(gaussians, K_test, target_p)

ax = axes[1]
ax.imshow(target_img.permute(1, 2, 0).numpy())
ax.set_title('Target (GT)', fontsize=11, fontweight='bold')
ax.axis('off')

ax = axes[2]
ax.imshow(rendered[0].permute(1, 2, 0).numpy())
ax.set_title('Rendered', fontsize=11, fontweight='bold')
ax.axis('off')

plt.suptitle('Training Results (Toy Example)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Note: This is a minimal demo with synthetic data and simple renderer.")
print("Real training uses gsplat rendering and RE10K/ACID datasets.")

## 5. Evaluation Metrics

### 5.1 Standard Metrics

| Metric | Range | Better | Measures |
|--------|-------|--------|----------|
| **PSNR** | [0, +inf) dB | Higher | Pixel-level accuracy |
| **SSIM** | [0, 1] | Higher | Structural similarity |
| **LPIPS** | [0, 1+] | Lower | Perceptual quality |

In [ ]:
def compute_psnr(predicted, target, max_val=1.0):
    """Compute PSNR (Peak Signal-to-Noise Ratio) in dB."""
    mse = (predicted - target).pow(2).mean()
    if mse < 1e-10:
        return torch.tensor(float('inf'))
    return 10 * torch.log10(max_val ** 2 / mse)


def compute_ssim_metric(predicted, target):
    """Compute SSIM as an evaluation metric."""
    return compute_ssim(predicted, target).mean()


# Evaluate at different quality levels
torch.manual_seed(42)
target = torch.rand(1, 3, 64, 64)

quality_levels = [
    ('Perfect', target.clone()),
    ('Low noise', target + torch.randn_like(target) * 0.02),
    ('Medium noise', target + torch.randn_like(target) * 0.1),
    ('High noise', target + torch.randn_like(target) * 0.3),
    ('Random', torch.rand_like(target)),
]

print(f"{'Quality':15s} | {'PSNR (dB)':>10s} | {'SSIM':>8s} | Interpretation")
print("-" * 65)

for name, pred in quality_levels:
    pred = pred.clamp(0, 1)
    psnr = compute_psnr(pred, target).item()
    ssim_val = compute_ssim_metric(pred, target).item()

    if psnr > 50:
        interp = 'Excellent'
    elif psnr > 30:
        interp = 'Good'
    elif psnr > 20:
        interp = 'Acceptable'
    else:
        interp = 'Poor'

    psnr_str = f'{psnr:.1f}' if psnr < 100 else 'inf'
    print(f"{name:15s} | {psnr_str:>10s} | {ssim_val:>8.4f} | {interp}")

print("\nBenchmark ranges for feed-forward methods on RE10K:")
print("  PSNR: 24-28 dB (higher is better)")
print("  SSIM: 0.85-0.92 (higher is better)")
print("  LPIPS: 0.10-0.20 (lower is better)")

## 6. Training Strategies

### 6.1 Learning Rate Schedule

Both MVSplat and pixelSplat use warm-up + cosine decay:

```
LR ────╮
       │╲
       │ ╲──────────────╮
       │                 ╲
       │                  ╲___________
       └──────────────────────────────── steps
       warmup     training      decay
```

### 6.2 Training Tips

In [ ]:
# Visualize learning rate schedule

def warmup_cosine_lr(step, total_steps, warmup_steps, base_lr, min_lr=1e-6):
    """Warmup + cosine decay learning rate schedule."""
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * progress))


total_steps = 300000
warmup_steps = 2000
base_lr = 1e-4

steps = np.arange(total_steps)
lrs = [warmup_cosine_lr(s, total_steps, warmup_steps, base_lr) for s in steps]

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(steps, lrs, lw=1.5, color='#1565C0')
ax.axvline(x=warmup_steps, color='red', linestyle='--', alpha=0.5, label='End warmup')
ax.set_xlabel('Training Step', fontsize=11)
ax.set_ylabel('Learning Rate', fontsize=11)
ax.set_title('Warmup + Cosine Decay Schedule', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

strategies = """
Key Training Strategies:

1. GRADIENT CLIPPING: clip to max_norm=1.0 (prevent exploding gradients)
2. MIXED PRECISION: FP16 for speed, FP32 for critical ops (depth, loss)
3. VIEW SAMPLING: random context/target pairs each step
4. DATA AUGMENTATION: color jitter, random crop, horizontal flip
5. MULTI-SCALE: train at multiple resolutions (128 -> 256)
6. BATCH SIZE: 4-8 scenes per GPU (limited by Cost Volume memory)
"""
print(strategies)

## 7. Supervised vs Self-supervised Training

| Approach | Supervision | Pros | Cons |
|----------|------------|------|------|
| **Self-supervised** | Novel view photometric loss | No depth GT needed, works on video data | Harder to train, slow convergence |
| **Depth supervised** | + Ground truth depth | Faster convergence, better geometry | Needs depth sensor or estimation |
| **Hybrid** | Photometric + pseudo-depth | Best of both worlds | More complex pipeline |

Most feed-forward methods use **self-supervised** training (photometric loss only), which is why they can train on large video datasets like RE10K.

In [ ]:
# Summary

summary = """
=====================================================================
   Notebook 05 Summary: Feed-forward Training & Loss Design
=====================================================================

1. TRAINING PARADIGM
   - Train on 1000s of scenes, test on new scenes
   - Input: 2 context views → predict Gaussians
   - Loss: render novel view and compare to ground truth

2. LOSS FUNCTIONS
   L_total = L1 + 0.2 * (1 - SSIM) + 0.05 * LPIPS
   - L1: pixel-level color accuracy (sharp images)
   - SSIM: structural similarity (perceptual quality)
   - LPIPS: deep perceptual metric (feature-level)

3. DATASETS
   - RE10K: primary benchmark (YouTube videos)
   - ACID: outdoor aerial scenes
   - DTU: controlled evaluation set

4. TRAINING LOOP
   context images → encoder → geometry → predict → render → loss
   Gradient flows end-to-end through differentiable rendering

5. EVALUATION METRICS
   - PSNR: 24-28 dB (higher = better)
   - SSIM: 0.85-0.92 (higher = better)
   - LPIPS: 0.10-0.20 (lower = better)

6. STRATEGIES
   - Warmup + cosine LR decay
   - Gradient clipping, mixed precision
   - Multi-scale training
   - Self-supervised (no depth GT needed)

=====================================================================
"""
print(summary)

## What's Next?

**[06_mvsplat_code_walkthrough.ipynb](./06_mvsplat_code_walkthrough.ipynb)** - Official MVSplat code walkthrough: repository structure, model implementation details, and running inference.

---

## References

1. MVSplat: https://arxiv.org/abs/2403.14627
2. pixelSplat: https://arxiv.org/abs/2312.12337
3. RE10K: https://google.github.io/realestate10k/
4. SSIM: https://en.wikipedia.org/wiki/Structural_similarity
5. LPIPS: https://arxiv.org/abs/1801.03924